In [ ]:
import functools
import operator
from typing import Any, Callable, Iterable, Sequence, Tuple, Union, Optional
import os
import sys

os.environ["JAX_PLATFORMS"] = "cpu"
sys.path.append('/home/lishengping/projects/maxtext/MaxText/')

import flax
import flax.linen as nn
import jax
from jax import lax
import jax.numpy as jnp
import common_types
from layers import initializers
from layers import normalizations
from layers import quantizations
import numpy as np
from jax.ad_checkpoint import checkpoint_name
from jax.experimental import shard_map
import math
import max_logging
import max_utils
from aqt.jax.v2 import aqt_tensor
from kernels import megablox as mblx


In [ ]:
def permute_new(inputs, gate_logits):
    inputs_shape = inputs.shape
    inputs_2d = jnp.reshape(inputs, (inputs_shape[0] * inputs_shape[1], inputs_shape[2]))
    weights, selected_experts = jax.lax.top_k(gate_logits, num_experts_per_tok)
    weights = jax.nn.softmax(weights.astype(jnp.float32), axis=-1).astype(dtype)
    flatten_selected_experts = jnp.ravel(selected_experts)
    permutation_indices = jnp.argsort(flatten_selected_experts)
    original_token_indices = permutation_indices // num_experts_per_tok
    sorted_inputs = jnp.take(inputs_2d, indices=original_token_indices, axis=0).astype(dtype)
    group_size = jnp.bincount(flatten_selected_experts, length=num_experts)
    print(f'permute_new....')
    return sorted_inputs, permutation_indices, original_token_indices, weights, group_size

  
def unpermute_new(intermediate, permutation_indices, original_token_indices, weights, batch_size, sequence_length):
    flat_weights = jnp.ravel(weights)
    permuted_flat_weights = jnp.take(flat_weights, indices=permutation_indices, axis=0)
    weighted_intermediate = intermediate.astype(jnp.float32) * permuted_flat_weights[:, None].astype(jnp.float32)
    num_original_tokens = batch_size * sequence_length
    combined_output = jax.ops.segment_sum(
        data=weighted_intermediate,
        segment_ids=original_token_indices,
        num_segments=num_original_tokens
    )
    print(f'unpermute_new....')
    return combined_output.reshape(batch_size, sequence_length, -1).astype(dtype)
  
def permute(inputs, gate_logits):
    inputs_shape = inputs.shape
    inputs_2d = jnp.reshape(inputs, (inputs_shape[0] * inputs_shape[1], inputs_shape[2]))
    weights, selected_experts = jax.lax.top_k(gate_logits, num_experts_per_tok)
    weights = jax.nn.softmax(weights.astype(jnp.float32), axis=-1).astype(dtype)
    flatten_selected_experts = jnp.ravel(selected_experts)
    sorted_selected_experts = jnp.argsort(flatten_selected_experts)
    sorted_indices = sorted_selected_experts // num_experts_per_tok
    # inputs_2d: (b, d) sorted_indices: (topk*length, ), sorted_inputs: (topk*length, d)
    sorted_inputs = jnp.take(inputs_2d, indices=sorted_indices, axis=0).astype(dtype)
    group_size = jnp.bincount(flatten_selected_experts, length=num_experts)
    print(f'permute....')
    return sorted_inputs, sorted_selected_experts, weights, group_size

def unpermute(intermediate, sorted_selected_experts, weights, batch_size, sequence_length):
    unsort_intermediate = jnp.take(intermediate, indices=jnp.argsort(sorted_selected_experts), axis=0)
    reshaped_weights = jnp.reshape(weights, (-1, num_experts_per_tok))
    reshaped_intermediate = jnp.reshape(
        unsort_intermediate,
        (reshaped_weights.shape[0], num_experts_per_tok, -1),
    )
    with jax.named_scope("weight_sum"):
      matmul_precision = lax.Precision('default')
      output = jnp.einsum(
          "BKE,BK -> BE",
          reshaped_intermediate.astype(jnp.float32),
          reshaped_weights.astype(jnp.float32),
          precision=matmul_precision,
      )
    print(f'unpermute....')
    return output.reshape(batch_size, sequence_length, -1).astype(dtype)

def gmm(inputs, kernel, group_sizes):
  hs_shape = inputs.shape
  # pad length is the 1st dimension of tiling size in gmm call
  pad_length = 512
  if hs_shape[0] % pad_length:
    pad_length = pad_length - hs_shape[0] % pad_length
    inputs = jax.lax.pad(inputs.astype(jnp.float32), 0.0, [(0, pad_length, 0), (0, 0, 0)])

  inputs = inputs.astype(dtype)
  kernel = kernel.astype(dtype)

  lhs_quantize_dtype, rhs_quantize_dtype = None, None

  if 1:
    # inputs: 2d, (batch*length) * dim
    print(f'inputs: {inputs.shape} kernel: {kernel.shape}')
    m, k, n = inputs.shape[0], inputs.shape[1], kernel.shape[2]
    for kd in [512, 768, 384, 256, 128]:
      if n % kd == 0:
        break
    # if too large, will exceed vmem.
    tile_size = (min(1024, m), min(1024, k), min(kd, n)) # 3d suggest 384~768，2d suggest 512~1024，1d suggest 1024
    print(f'tile_size: {tile_size}')

    output = mblx.gmm(
        lhs=inputs,
        rhs=kernel,
        group_sizes=group_sizes,
        preferred_element_type=jnp.bfloat16,
        tiling=tile_size,
        lhs_quantize_dtype=lhs_quantize_dtype,
        rhs_quantize_dtype=rhs_quantize_dtype,
    )
  return output

def _convert_to_activation_function(fn_or_string: Union[str, Callable[..., Any]]) -> Callable[..., Any]:
  if fn_or_string == "linear":
    return lambda x: x
  elif isinstance(fn_or_string, str):
    return getattr(nn, fn_or_string)
  elif callable(fn_or_string):
    return fn_or_string
  else:
    raise ValueError(
        f"""Don't know how to convert {fn_or_string}
                         to an activation function"""
    )

import time

start = time.time()

b, l, d, m = 32, 256, 512, 512
num_experts = 16
num_experts_per_tok = 2
dtype = jnp.bfloat16
x = jax.random.uniform(jax.random.PRNGKey(0), (b, l, d))
logits = jax.random.uniform(jax.random.PRNGKey(0), (b, l, num_experts_per_tok))
w0 = jax.random.uniform(jax.random.PRNGKey(0), (num_experts, d, m))
w1 = jax.random.uniform(jax.random.PRNGKey(1), (num_experts, d, m))
wo = jax.random.uniform(jax.random.PRNGKey(2), (num_experts, m, d))

use_permute_new = True

batch_size, sequence_length, _ = x.shape
if use_permute_new:
    x, permutation_indices, original_token_indices, weights, group_sizes = permute_new(x, logits)
else:
    x, sorted_selected_experts, weights, group_sizes = permute(x, logits)
    
layer_w0 = gmm(x, w0, group_sizes)
layer_w1 = gmm(x, w1, group_sizes)
layer_act = _convert_to_activation_function('silu')(layer_w0)
intermediate_layer = jnp.multiply(layer_act, layer_w1)
intermediate_output = gmm(intermediate_layer, wo, group_sizes)
intermediate_output = checkpoint_name(intermediate_output, "mlpwo")
if use_permute_new:
    output = unpermute_new(
        intermediate_output, 
        permutation_indices,
        original_token_indices,
        weights, 
        batch_size=batch_size, 
        sequence_length=sequence_length
    )
    new = output
else:
    output = unpermute(
      intermediate_output, sorted_selected_experts, weights, batch_size=batch_size, sequence_length=sequence_length
    )
    old = output  

print(f'Finished.... take: {time.time() - start:.5f}s')